<a href="https://colab.research.google.com/github/jeffheaton/app_generative_ai/blob/main/t81_559_class_05_4_custom_parsers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-559: Applications of Generative Artificial Intelligence
**Module 5: LangChain: Data Extraction**
* Instructor: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.wustl.edu/Programs/Pages/default.aspx)
* For more information visit the [class website](https://github.com/jeffheaton/app_generative_ai).

# Module 5 Material

* Part 5.1: The Structured Output Problem [[Notebook]](t81_559_class_05_1_langchain_data.ipynb)
* Part 5.2: Designing Schemas with Pydantic [[Notebook]](t81_559_class_05_2_parsers.ipynb)
* Part 5.3: Validation, Retries, and Refusals [[Notebook]](t81_559_class_05_3_pydantic.ipynb)
* **Part 5.4: Structured Extraction at Scale** [[Notebook]](t81_559_class_05_4_custom_parsers.ipynb)
* Part 5.5: Structured Output Under the Hood [[Notebook]](t81_559_class_05_5_output_fixing_parsers.ipynb)

# Google CoLab Instructions

The following code ensures that Google CoLab is running and maps Google Drive if needed.

In [2]:
import os

try:
    from google.colab import drive, userdata
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# OpenAI Secrets
if COLAB:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Install needed libraries in CoLab
if COLAB:
    !pip install langchain langchain_openai

Note: using Google CoLab
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


# 5.4: Structured Extraction at Scale

The single most common production use of LLMs is not chat. It is *extraction*: taking a pile of unstructured documents -- contracts, support tickets, clinical notes, resumes -- and turning them into a clean dataset that ordinary software can query, join, and chart. Everything you learned in Parts 5.1 through 5.3 assembles into that pipeline here.

Our corpus is one you will meet again in Module 7: five hundred randomly generated biographical sketches of employees at five fictional companies, hosted with the course data. Each biography is a paragraph of prose. By the end of this notebook it will be a pandas DataFrame.

In [3]:
from langchain_openai import ChatOpenAI

MODEL = 'gpt-5.6-luna'

llm = ChatOpenAI(model=MODEL)

## Loading the Corpus

Each file holds one hundred biographies separated by blank lines. The following code downloads all five files and splits them into individual documents.

In [4]:
import requests

urls = [
    "https://data.heatonresearch.com/data/t81-559/bios/DD.txt",
    "https://data.heatonresearch.com/data/t81-559/bios/FT.txt",
    "https://data.heatonresearch.com/data/t81-559/bios/GS.txt",
    "https://data.heatonresearch.com/data/t81-559/bios/NGS.txt",
    "https://data.heatonresearch.com/data/t81-559/bios/TI.txt",
]

bios_by_file = {}
for url in urls:
    text = requests.get(url).text
    name = url.rsplit("/", 1)[-1]
    bios_by_file[name] = [b.strip() for b in text.split("\n\n") if b.strip()]
    print(f"{name}: {len(bios_by_file[name])} biographies")

sample_bio = bios_by_file["DD.txt"][0]
print("\n--- example biography ---\n")
print(sample_bio)

DD.txt: 100 biographies
FT.txt: 100 biographies
GS.txt: 100 biographies
NGS.txt: 100 biographies
TI.txt: 100 biographies

--- example biography ---

Samantha Clarke is a seasoned Project Manager at Digital Dynamics, a leading tech company known for its innovative approach to digital transformation solutions. With over a decade of experience in the tech industry, Samantha has played a pivotal role in steering complex projects to success, specializing in integrating artificial intelligence with traditional business processes to enhance efficiency and profitability. A graduate of MIT with a degree in Computer Science, she has a keen interest in emerging technologies and their potential to solve real-world business challenges. Outside of her professional life, Samantha is an avid rock climber and enjoys mentoring young women interested in STEM careers, aiming to inspire and cultivate a new generation of tech leaders.


## Designing the Extraction Schema

The schema applies every lesson from Part 5.2. Values the biography must contain (a name, a job title, an employer) are required; values that may or may not appear (a university, years of experience) are `Optional`, which is our instruction to the model not to guess; and hobbies are a list that may legitimately be empty.

In [5]:
from typing import Optional
from pydantic import BaseModel, Field

class PersonRecord(BaseModel):
    """Structured facts about one employee, extracted from a biography."""
    name: str = Field(description="The person's full name")
    job_title: str = Field(description="Their current job title")
    employer: str = Field(description="The company they currently work for")
    university: Optional[str] = Field(default=None, description="University they attended, only if named in the text")
    years_experience: Optional[int] = Field(default=None, description="Years of professional experience, only if stated")
    hobbies: list[str] = Field(default_factory=list, description="Hobbies or personal interests mentioned, empty if none")

structured_llm = llm.with_structured_output(PersonRecord)

person = structured_llm.invoke(sample_bio)
print(person)

name='Samantha Clarke' job_title='Project Manager' employer='Digital Dynamics' university='MIT' years_experience=10 hobbies=['Emerging technologies', 'Rock climbing', 'Mentoring young women interested in STEM careers']


One paragraph in, one typed record out. Now we do it five hundred times -- or rather, we let LangChain do it concurrently.

## Batch Extraction

Every LangChain runnable has a `.batch()` method that processes a list of inputs in parallel, which matters when each call takes a second or two of network time. We limit concurrency to be polite to the API. To keep this classroom run fast we extract the first ten biographies from each company (fifty records); scaling to all five hundred is the same line of code with a longer coffee break.

In [6]:
sample = []
for name, bios in bios_by_file.items():
    sample.extend(bios[:10])

print(f"Extracting {len(sample)} biographies...")
people = structured_llm.batch(sample, config={"max_concurrency": 8})
print(f"Extracted {len(people)} records")

for p in people[:3]:
    print(f"  {p.name:<22} {p.job_title:<28} {p.employer}")

Extracting 50 biographies...
Extracted 50 records
  Samantha Clarke        Project Manager              Digital Dynamics
  Samantha Clarke        Project Manager              Digital Dynamics
  Samantha Clarke        Software Engineer            Digital Dynamics


## From Objects to a Dataset

A list of Pydantic objects converts to a DataFrame in one line, because `model_dump()` turns each record into a plain dictionary. From there, it is ordinary data analysis -- the LLM's work is done.

In [7]:
import pandas as pd

df = pd.DataFrame([p.model_dump() for p in people])
df.head()

,name,job_title,employer,university,years_experience,hobbies
0,Samantha Clarke,Project Manager,Digital Dynamics,MIT,10,"[Emerging technologies, Rock climbing, Mentori..."
1,Samantha Clarke,Project Manager,Digital Dynamics,MIT,10,"[rock climbing, teaching coding to young girls]"
2,Samantha Clarke,Software Engineer,Digital Dynamics,MIT,10,"[Rock climbing, Mentoring young women interest..."
3,Jason Carter,Software Engineer,Digital Dynamics,MIT,10,"[rock climbing, volunteering to teach coding t..."
4,Jason Carter,Software Engineer,Digital Dynamics,MIT,10,"[coding, rock climbing, teaching coding to und..."


In [8]:
print("Employees per company:")
print(df["employer"].value_counts())

print("\nMost common hobbies:")
print(df.explode("hobbies")["hobbies"].value_counts().head(8))

print(f"\nBios stating years of experience: {df['years_experience'].notna().sum()} of {len(df)}")

Employees per company:
employer
Digital Dynamics    10
FutureTech          10
Global Solutions    10
NextGen Software    10
Tech Innovators     10
Name: count, dtype: int64

Most common hobbies:
hobbies
rock climbing                                              22
Rock climbing                                              18
Mentoring young women interested in STEM careers           10
mentoring young tech enthusiasts                            8
mentoring young tech professionals                          2
Emerging technologies                                       2
Mentoring young aspiring engineers                          2
volunteering teaching robotics to underprivileged youth     2
Name: count, dtype: int64

Bios stating years of experience: 50 of 50


## What Did That Cost?

A biography runs roughly 200 tokens, and with the schema and response the extraction of all fifty records consumed on the order of 25,000 tokens. At gpt-5.6-luna's pricing ($0.20 per million input tokens, $1.20 per million output tokens), this entire demonstration cost well under one cent, and extracting all five hundred biographies would cost roughly a dime. This economics is *why* extraction is the dominant production use of LLMs: work that once required a data-entry team now costs pocket change.

Two production notes before you scale a pipeline like this. First, wrap each call with the `safe_extract` pattern from Part 5.3 so one bad record cannot kill a long run, and keep the errors as a column for triage. Second, spot-check a sample of the output against the source text -- extraction quality is measured, not assumed. (Formal evaluation of LLM pipelines is a topic we return to later in the course.)

## Summary

The pattern you just built -- **schema, batch, DataFrame** -- is the bread and butter of applied LLM engineering. The model touches the data exactly once, and everything downstream is ordinary, testable Python. In the final part of this module, we open the hood: how does the API actually force a model to produce schema-shaped output, and what do you do with models that cannot?